## Clone the repo + enter the dir

In [ ]:
# Go back to root
%cd /content


# Clone fresh
!git clone https://github.com/zohir22s/cifar10-ann-cnn-comparison.git

# Enter the directory
%cd cifar10-ann-cnn-comparison

# Create the missing __init__.py files
!touch data/__init__.py
!touch models/__init__.py
!touch training/__init__.py
!touch evaluation/__init__.py
# Test imports
from data.load_data import train_loader, test_loader
from models.ann_model import ANN
print("✅ All imports successful!")

Install req (Optional)

In [ ]:
# !pip install -r requirements.txt

## Imports

In [ ]:
import torch
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score, confusion_matrix

## Training

In [ ]:
%run training/train_ann.py


In [ ]:
#%run training/train_cnn.py

In [ ]:
#%run training/train_advanced.py

## Load Trained Models and Collect Predictions


### ANN:

In [ ]:
from models.ann_model import ANN
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ann_model = ANN.from_pretrained("/content/cifar10-ann-cnn-comparison/results/models/ann_best.pth", device)

Predict :

In [ ]:
ann_preds = []

with torch.no_grad():
    for img in X_test_norm:
        tensor = torch.tensor(img).permute(2,0,1).unsqueeze(0).to(device)
        output = ann(tensor)
        pred = torch.argmax(output, dim=1).item()
        ann_preds.append(pred)

ann_preds = np.array(ann_preds)

### CNN:

In [ ]:
cnn = tf.keras.models.load_model("results/models/cnn_adam.h5")

Predict :

In [ ]:
cnn_preds = np.argmax(cnn.predict(X_test_norm), axis=1)

### Advanced CNN:

In [ ]:
from models.cnn_advanced import CNN_Advanced

cnn_adv = CNN_Advanced().to(device)
cnn_adv.load_state_dict(torch.load("results/models/cnn_advanced_best.pth", map_location=device))
cnn_adv.eval()

Predict :

In [ ]:
cnn_adv_preds = []

with torch.no_grad():
    for img in X_test_norm:
        tensor = torch.tensor(img).permute(2,0,1).unsqueeze(0).to(device)
        output = cnn_adv(tensor)
        pred = torch.argmax(output, dim=1).item()
        cnn_adv_preds.append(pred)

cnn_adv_preds = np.array(cnn_adv_preds)

## Comparisons :

### Accuracy comparison :

In [ ]:
ann_acc = accuracy_score(y_test, ann_preds)
cnn_acc = accuracy_score(y_test, cnn_preds)
cnn_adv_acc = accuracy_score(y_test, cnn_adv_preds)

print("ANN Accuracy:", ann_acc)
print("CNN Accuracy:", cnn_acc)
print("CNN Advanced Accuracy:", cnn_adv_acc)

### Bar chart comparison

In [ ]:
models = ["ANN", "CNN", "CNN Advanced"]
accuracies = [ann_acc, cnn_acc, cnn_adv_acc]

plt.figure(figsize=(6,4))
plt.bar(models, accuracies)
plt.title("Model Comparison on CIFAR-10")
plt.ylabel("Accuracy")
plt.show()

### Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, cnn_adv_preds)

plt.imshow(cm)
plt.title("CNN Advanced Confusion Matrix")
plt.colorbar()
plt.show()